# KalaKriti — Stage 5A: Warli Neural Style Transfer Pilot
### Real-Time Multi-Style Neural Style Transfer for Visual Domain Adaptation and Indian Traditional Art

This notebook executes the **Stage 5A Pilot Workflow**:
1. **Environment & GPU Verification** (Ensures active CUDA GPU)
2. **Project Code & Repository Setup** (Interactive upload if missing)
3. **Dependency Installation** (Without replacing Colab's CUDA PyTorch)
4. **Google Drive Mount & Path Configuration**
5. **Kaggle Dataset Acquisition & Deep Inspection** (`ajg117/indian-paintings-dataset`)
6. **Warli Art Verification & Mandatory Contact Sheet Approval**
7. **Content & Unseen Validation Acquisition & Integrity Validation**
8. **Pre-Flight Dataset Validation** (`backend/validate_dataset.py` via Python subprocess)
9. **Pilot Training Run** (`backend/run_colab_pilot.py`, 2 epochs, batch size 4, seed 42)
10. **Visual Inspection Plate Evaluation** (Side-by-side 3-panel review)
11. **Checkpoint Verification & FastAPI Server + Secure ngrok Deployment**

--- 
## Cell 1: Check Colab Environment & GPU Capabilities
Verifies Python, PyTorch, and CUDA device allocation. Halts execution if running on CPU.

In [ ]:
import sys
import torch
import torchvision

print("=" * 65)
print("KALAKRITI SYSTEM & ENVIRONMENT CHECK")
print("=" * 65)
print(f"Python Version:      {sys.version.split()[0]}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

cuda_available = torch.cuda.is_available()
print(f"CUDA Available:      {cuda_available}")

if cuda_available:
    gpu_count = torch.cuda.device_count()
    device_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Devices:         {gpu_count} ({device_name})")
    print(f"GPU VRAM Memory:     {total_mem_gb:.2f} GB")
    print("✓ Environment check passed. Suitable GPU allocated.")
else:
    print("\n" + "!" * 65)
    print("❌ WARNING: No GPU detected! Current runtime is CPU-only.")
    print("   Please switch runtime: Runtime -> Change runtime type -> T4 GPU.")
    print("!" * 65)
    raise SystemExit("Stopping: GPU runtime is required for neural style transfer training.")

--- 
## Cell 2: Verify Backend Project Files & Interactive Uploader
Checks for required backend files. If missing, launches an interactive in-notebook uploader to extract `kalakriti_colab_package.zip`.

In [ ]:
import os
import zipfile

required_files = [
    "backend/models/transformer_net.py",
    "backend/train_colab.py",
    "backend/run_colab_pilot.py",
    "backend/validate_dataset.py",
    "backend/server.py",
    "backend/requirements.txt",
]

# Check if backend is available in current or sub-directories
if not os.path.exists("backend/models/transformer_net.py"):
    if os.path.exists("/content/KalaKriti/backend"):
        os.chdir("/content/KalaKriti")
    elif os.path.exists("/content/drive/MyDrive/KalaKriti/backend"):
        os.chdir("/content/drive/MyDrive/KalaKriti")

# If still missing, launch interactive upload
if not os.path.exists("backend/models/transformer_net.py"):
    print("Backend files not found in working directory.")
    print("Please upload 'kalakriti_colab_package.zip':")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            with zipfile.ZipFile(fname, 'r') as zf:
                zf.extractall(".")
            print(f"✓ Extracted {fname} successfully.")

# Verify all required files
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"❌ Critical backend files missing: {missing}. Ensure the backend folder is extracted.")

print("✓ All required KalaKriti backend files verified:")
for f in required_files:
    print(f"  - {f}")

--- 
## Cell 3: Install Required Dependencies
Installs `kagglehub`, `fastapi`, `uvicorn`, `pyngrok`, `python-multipart`, and `httpx` without replacing Colab's CUDA PyTorch.

In [ ]:
!pip install -q kagglehub fastapi uvicorn pyngrok python-multipart pillow pydantic requests tqdm httpx

try:
    import kagglehub
    import fastapi
    import uvicorn
    import pyngrok
    import PIL
    print("✓ All required runtime dependencies imported successfully.")
except ImportError as e:
    raise ImportError(f"❌ Dependency import failure: {e}")

--- 
## Cell 4: Mount Google Drive & Configure Directory Paths
Connects Google Drive to persist training datasets, checkpoints, and evaluation plates.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Configurable paths
BASE_DRIVE_DIR  = "/content/drive/MyDrive/KalaKriti"
WARLI_TRAIN_DIR = os.path.join(BASE_DRIVE_DIR, "data/warli_train")
COCO_TRAIN_DIR  = os.path.join(BASE_DRIVE_DIR, "data/coco_train")
VAL_CONTENT_DIR = os.path.join(BASE_DRIVE_DIR, "data/val_content")
PILOT_OUT_DIR   = os.path.join(BASE_DRIVE_DIR, "pilot_outputs")
CHECKPOINT_DIR  = os.path.join(BASE_DRIVE_DIR, "checkpoints")

for p in [WARLI_TRAIN_DIR, COCO_TRAIN_DIR, VAL_CONTENT_DIR, PILOT_OUT_DIR, CHECKPOINT_DIR]:
    os.makedirs(p, exist_ok=True)

print("✓ Directory paths configured:")
print(f"  - Warli Training Art:   {WARLI_TRAIN_DIR}")
print(f"  - Content Photographs:  {COCO_TRAIN_DIR}")
print(f"  - Validation Images:    {VAL_CONTENT_DIR}")
print(f"  - Saved Checkpoints:    {CHECKPOINT_DIR}")
print(f"  - Inspection Plates:    {PILOT_OUT_DIR}")

--- 
## Cell 5: Download & Recursively Inspect Kaggle Dataset
Downloads `ajg117/indian-paintings-dataset` (MIT License) and audits its file structure and categories.

In [ ]:
import kagglehub
import os
from collections import defaultdict

print("Downloading ajg117/indian-paintings-dataset via kagglehub...")
try:
    dataset_path = kagglehub.dataset_download("ajg117/indian-paintings-dataset")
    print(f"✓ Download completed. Local path: {dataset_path}")
except Exception as e:
    print(f"❌ Error downloading dataset: {e}")
    print("If Kaggle credentials are needed:")
    print("1. Go to Kaggle -> Settings -> Create New Token (kaggle.json)")
    print("2. Upload kaggle.json to ~/.kaggle/kaggle.json and chmod 600")
    raise e

print("\n" + "=" * 65)
print("RECURSIVE STRUCTURAL AUDIT OF DATASET")
print("=" * 65)

all_files = []
ext_counts = defaultdict(int)
folder_counts = defaultdict(int)

for root, dirs, files in os.walk(dataset_path):
    rel_root = os.path.relpath(root, dataset_path)
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        ext_counts[ext] += 1
        all_files.append(os.path.join(root, f))
        folder_counts[rel_root] += 1

print(f"Total Files Found: {len(all_files)}")
print("Subdirectories Detected:")
for folder, count in sorted(folder_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  - {folder}: {count} files")

--- 
## Cell 6: Identify Warli Artworks, Generate Contact Sheet & Require Human Approval
Locates Warli images, displays an inline 12-image contact sheet, and prompts for explicit human approval before copying to `WARLI_TRAIN_DIR`.

In [ ]:
import shutil
import hashlib
from PIL import Image
import matplotlib.pyplot as plt

def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

# Search specifically for Warli category files
warli_candidates = []
allowed_img_exts = (".jpg", ".jpeg", ".png", ".webp")

for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith(allowed_img_exts):
            full_path = os.path.join(root, f)
            if "warli" in full_path.lower():
                warli_candidates.append(full_path)

print(f"Candidate Warli Images Found: {len(warli_candidates)}")

if len(warli_candidates) == 0:
    raise RuntimeError("❌ No Warli images found in dataset. Provide authentic Warli artwork images.")

# Display contact sheet for human inspection
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()
for i in range(12):
    if i < len(warli_candidates):
        img = Image.open(warli_candidates[i]).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(warli_candidates[i])[:15], fontsize=8)
    axes[i].axis("off")
plt.suptitle("Warli Candidate Visual Review (Verify white stick figures on ochre backdrop)", fontsize=12)
plt.tight_layout()
plt.show()

# MANDATORY HUMAN CONFIRMATION
print("=" * 65)
print("HUMAN VISUAL VERIFICATION REQUIRED")
print("=" * 65)
print("Inspect the contact sheet. Confirm paintings exhibit authentic Warli grammar (white figures on ochre/earthen background).")
user_approval = input("Do the candidate images depict authentic Warli art? Type 'YES' to approve: ").strip()

if user_approval.upper() != "YES":
    raise SystemExit("Stopping: User did not confirm candidate artwork authenticity.")

# Deduplicate and copy verified artworks
seen_hashes = set()
copied_count = 0
for src in warli_candidates[:20]:
    file_hash = compute_sha256(src)
    if file_hash not in seen_hashes:
        seen_hashes.add(file_hash)
        dst = os.path.join(WARLI_TRAIN_DIR, f"warli_{copied_count+1:03d}{os.path.splitext(src)[1]}")
        shutil.copy2(src, dst)
        copied_count += 1

print(f"✓ Copied {copied_count} verified unique Warli artworks into: {WARLI_TRAIN_DIR}")

--- 
## Cell 7: Acquire Content Training Photos & 4 Unseen Validation Benchmarks
Downloads 50 MS-COCO 2017 natural photographs and 4 separate unseen benchmarks (`portrait.jpg`, `monument.jpg`, `landscape.jpg`, `animal.jpg`), verifying integrity with PIL.

In [ ]:
import urllib.request
from PIL import Image

# 1. 50 Content Photographs (MS-COCO 2017)
coco_ids = [
    139, 285, 632, 724, 776, 785, 802, 872, 885, 1000,
    1268, 1296, 1353, 1422, 1442, 1503, 1532, 1584, 1645, 1700,
    1721, 1761, 1818, 1888, 1928, 2010, 2045, 2088, 2125, 2144,
    2153, 2185, 2210, 2232, 2246, 2289, 2302, 2335, 2384, 2405,
    2426, 2467, 2482, 2521, 2534, 2582, 2605, 2642, 2688, 2700
]

downloaded_content = 0
for idx, cid in enumerate(coco_ids):
    fname = f"content_{idx+1:03d}.jpg"
    dest = os.path.join(COCO_TRAIN_DIR, fname)
    if not os.path.exists(dest) or os.path.getsize(dest) < 5120:
        url = f"http://images.cocodataset.org/val2017/000000{cid:06d}.jpg"
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=10) as resp, open(dest, 'wb') as f:
                f.write(resp.read())
        except Exception:
            pass

# Validate content images with PIL
valid_content_count = 0
for f in os.listdir(COCO_TRAIN_DIR):
    fpath = os.path.join(COCO_TRAIN_DIR, f)
    try:
        with Image.open(fpath) as img:
            img.verify()
        if os.path.getsize(fpath) > 5120:
            valid_content_count += 1
    except Exception:
        os.remove(fpath)

if valid_content_count < 50:
    raise RuntimeError(f"Expected at least 50 valid content photos, got {valid_content_count}.")

print(f"✓ Verified {valid_content_count} content training images in {COCO_TRAIN_DIR}")

# 2. Unseen Validation Benchmarks (Portrait, Monument, Landscape, Animal)
val_benchmarks = {
    "portrait.jpg": "https://images.unsplash.com/photo-1544005313-94ddf0286df2?auto=format&fit=crop&w=512&q=80",
    "monument.jpg": "https://images.unsplash.com/photo-1564507592333-c60657eea523?auto=format&fit=crop&w=512&q=80",
    "landscape.jpg": "https://images.unsplash.com/photo-1506744038136-46273834b3fb?auto=format&fit=crop&w=512&q=80",
    "animal.jpg": "https://images.unsplash.com/photo-1557050543-4d5f4e07ef46?auto=format&fit=crop&w=512&q=80",
}

for fname, url in val_benchmarks.items():
    dest = os.path.join(VAL_CONTENT_DIR, fname)
    if not os.path.exists(dest) or os.path.getsize(dest) < 5120:
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=10) as resp, open(dest, 'wb') as f:
                f.write(resp.read())
        except Exception as e:
            print(f"Warning downloading {fname}: {e}")

print(f"✓ Unseen validation benchmarks ready in {VAL_CONTENT_DIR}")

--- 
## Cell 8: Pre-Flight Dataset Validation via Python Subprocess
Executes `backend/validate_dataset.py` with native Python variable passing. Guarantees execution halts if validation fails.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "backend/validate_dataset.py",
    "--content_dir", COCO_TRAIN_DIR,
    "--style_dir", WARLI_TRAIN_DIR,
    "--val_dir", VAL_CONTENT_DIR
]

print(f"Executing: {' '.join(cmd)}")
# check=True ensures notebook stops immediately if exit code != 0
subprocess.run(cmd, check=True)

--- 
## Cell 9: Run the Warli Pilot Training Experiment (Seed 42)
Executes `backend/run_colab_pilot.py` targeting 50 content photographs, 10–20 Warli references, 2 epochs, batch size 4, and seed 42.

In [ ]:
cmd_pilot = [
    sys.executable, "backend/run_colab_pilot.py",
    "--content_dir", COCO_TRAIN_DIR,
    "--style_dir", WARLI_TRAIN_DIR,
    "--val_dir", VAL_CONTENT_DIR,
    "--checkpoint_dir", CHECKPOINT_DIR,
    "--output_dir", PILOT_OUT_DIR,
    "--max_samples", "50",
    "--epochs", "2",
    "--batch_size", "4",
    "--seed", "42"
]

print(f"Executing Pilot Run: {' '.join(cmd_pilot)}")
subprocess.run(cmd_pilot, check=True)

--- 
## Cell 10: Visual Evaluation of Pilot Inspection Plates
Renders the generated 3-panel comparison plates: `[Original Content | Warli Reference | KalaKriti Stylized]` inline in Colab.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

plate_files = sorted(glob.glob(os.path.join(PILOT_OUT_DIR, "*.jpg")))
print(f"Generated Comparison Plates Found: {len(plate_files)}")

for pf in plate_files:
    plate_img = Image.open(pf)
    plt.figure(figsize=(15, 5))
    plt.imshow(plate_img)
    plt.title(f"Evaluation Plate: {os.path.basename(pf)}\n[Left: Content | Center: Warli Style Reference | Right: KalaKriti Stylized]", fontsize=10)
    plt.axis("off")
    plt.show()

--- 
## Cell 11: Checkpoint Verification & FastAPI Server Launch with ngrok
Loads `kalakriti_cin_warli_pilot.pth` into `TransformerNet`, verifies compatibility, starts the FastAPI server, and launches an ngrok tunnel.

In [ ]:
import subprocess
import time
import shutil
import getpass
from pyngrok import ngrok
import requests

# 1. Verify Checkpoint Loading in TransformerNet
from backend.models.transformer_net import TransformerNet

pilot_ckpt_path = os.path.join(CHECKPOINT_DIR, "kalakriti_cin_warli_pilot.pth")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = TransformerNet(num_styles=5, base_channels=32)
state_dict = torch.load(pilot_ckpt_path, map_location=device)
net.load_state_dict(state_dict)
net.to(device).eval()
print(f"✓ Checkpoint successfully loaded: {pilot_ckpt_path}")

# Copy checkpoint into backend/checkpoints so backend/server.py finds it locally
os.makedirs("backend/checkpoints", exist_ok=True)
shutil.copy2(pilot_ckpt_path, "backend/checkpoints/kalakriti_cin_warli_pilot.pth")

# 2. Launch FastAPI Server in background
print("Starting FastAPI server...")
server_proc = subprocess.Popen(["uvicorn", "backend.server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# 3. Configure ngrok token securely without hardcoding
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    ngrok_token = None

if not ngrok_token:
    ngrok_token = getpass.getpass("Enter your ngrok Authtoken (leave blank if already configured in system): ").strip()

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000).public_url
print("\n" + "=" * 65)
print("KALAKRITI LIVE FASTAPI INFERENCE BACKEND READY")
print("=" * 65)
print(f"Public ngrok URL: {public_url}")
print("Paste this URL into the KalaKriti React frontend Settings Modal.")
print("=" * 65)

# 4. Test GET /health endpoint locally
res = requests.get("http://localhost:8000/health")
print("\nLocal /health ping response:")
print(res.json())